In [1]:
import json
import re
import chess
from pathlib import Path
from typing import List, Optional
from testing.board import visual_to_fen

In [2]:
def inspect_lines(file_path: str, line_numbers: List[int]) -> None:
    """
    Print the full chat entry (system / user / assistant) for each requested line
    in a .jsonl file.

    Parameters
    ----------
    file_path : str
        Path to the .jsonl file.
    line_numbers : List[int]
        1‑based line numbers to inspect.
    """
    targets = set(line_numbers)
    file_path = Path(file_path)

    with file_path.open("r", encoding="utf-8") as f:
        for idx, line in enumerate(f, start=1):
            if idx not in targets:
                continue

            try:
                obj = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"✗ {file_path}:{idx} JSON error -> {e}")
                continue

            chat = obj.get("chat", [])
            print(f"\n── {file_path}:{idx} ─────────────────────────────────────────────────")
            for role, msg in chat:
                print(f"{role.upper():9}: {msg}")
            print("─────────────────────────────────────────────────────────────")
            targets.remove(idx)

            if not targets:  # all requested lines printed
                break

    if targets:  # lines not found
        missing = ", ".join(map(str, sorted(targets)))
        print(f"\n[WARN] Lines not found in {file_path}: {missing}")

# Testing Correctness of `is_check` Samples

In [3]:
IS_CHECK_REGEX = re.compile(r"Is the (black|white) king in check\?")

def king_in_check(board: chess.Board, color: chess.Color) -> bool:
    """
    Return True iff the king of the given color is in check, independent of side‑to‑move.
    """
    king_sq = board.king(color)
    if king_sq is None:
        raise ValueError("No king for color on this board")
    return board.is_attacked_by(not color, king_sq)   # opponents attack the king square

def test_is_check(files: List[str], print_errors: bool = False) -> None:
    total = passed = failed = 0

    for file_path in files:
        file_path = Path(file_path)
        with file_path.open(encoding="utf-8") as fh:
            for line_num, line in enumerate(fh, 1):
                obj = json.loads(line)
                user_text        = obj["chat"][1][1]
                assistant_answer = obj["chat"][2][1].strip()

                m = IS_CHECK_REGEX.search(user_text)
                if not m:
                    print(f"[WARN] no question line {line_num} in {file_path}")
                    continue
                color_str  = m.group(1)                      # 'black' | 'white'
                color_bool = chess.BLACK if color_str == "black" else chess.WHITE

                fen   = visual_to_fen(user_text)
                board = chess.Board(fen)

                correct_answer = "Yes" if king_in_check(board, color_bool) else "No"

                total += 1
                if assistant_answer == correct_answer:
                    passed += 1
                else:
                    failed += 1
                    if print_errors:
                        print(f"[FAIL] {file_path}:{line_num}  expected {correct_answer}, got {assistant_answer}")

    print(f"[is_check] {passed}/{total} passed, {failed} failed")

# Testing Correctness of `mat_bal` and `large_mat_adv` Samples

In [4]:
# ────────────────────────────────────────────────────────────────
# Shared helpers
# ────────────────────────────────────────────────────────────────
_PIECE_VALUE = {
    chess.PAWN:   100,
    chess.KNIGHT: 320,
    chess.BISHOP: 330,
    chess.ROOK:   500,
    chess.QUEEN:  900,
    chess.KING:     0,
}

def material_cp(board: chess.Board) -> int:
    """Centipawn material score (white‑positive, black‑negative)."""
    return sum(
        _PIECE_VALUE[p] *
        (len(board.pieces(p, chess.WHITE)) - len(board.pieces(p, chess.BLACK)))
        for p in _PIECE_VALUE
    )

# ────────────────────────────────────────────────────────────────
# 1) “Is the game materially balanced?”
# ────────────────────────────────────────────────────────────────
MAT_BAL_REGEX = re.compile(r"Is the game materially balanced\?", re.I)

def test_mat_bal(files: List[str], print_errors: bool = False) -> None:
    total = passed = failed = 0

    for fpath in files:
        for ln, line in enumerate(Path(fpath).open(encoding="utf-8"), 1):
            obj           = json.loads(line)
            user_text     = obj["chat"][1][1]
            assistant_ans = obj["chat"][2][1].strip()

            if not MAT_BAL_REGEX.search(user_text):
                print(f"[WARN] No mat_bal question {fpath}:{ln}")
                continue

            board = chess.Board(visual_to_fen(user_text))
            balanced = abs(material_cp(board)) < 120     # same threshold the generator uses
            correct  = "Yes" if balanced else "No"

            total += 1
            if assistant_ans == correct:
                passed += 1
            else:
                failed += 1
                if print_errors:
                    print(f"[FAIL] {fpath}:{ln}  expected {correct}, got {assistant_ans}")

    print(f"[mat_bal] {passed}/{total} passed, {failed} failed")


# ────────────────────────────────────────────────────────────────
# 2) “Does <color> have a material advantage?”
#     (A *large* advantage means > 300 cp for that colour)
# ────────────────────────────────────────────────────────────────
LMA_REGEX   = re.compile(r"Does (white|black) have a material advantage\?", re.I)
LMA_THRESH  = 300   # centipawns

def test_large_mat_adv(files: List[str], print_errors: bool = False) -> None:
    total = passed = failed = 0

    for fpath in files:
        for ln, line in enumerate(Path(fpath).open(encoding="utf-8"), 1):
            obj           = json.loads(line)
            user_text     = obj["chat"][1][1]
            assistant_ans = obj["chat"][2][1].strip()

            m = LMA_REGEX.search(user_text)
            if not m:
                print(f"[WARN] No large_mat_adv question {fpath}:{ln}")
                continue

            ask_color = chess.WHITE if m.group(1).lower() == "white" else chess.BLACK

            board   = chess.Board(visual_to_fen(user_text))
            adv_cp  = material_cp(board)                # white‑positive
            # Check if the *queried* colour is ahead by more than 300 cp
            color_has_large_adv = (
                adv_cp >  LMA_THRESH and ask_color == chess.WHITE or
                adv_cp < -LMA_THRESH and ask_color == chess.BLACK
            )

            correct = "Yes" if color_has_large_adv else "No"

            total += 1
            if assistant_ans == correct:
                passed += 1
            else:
                failed += 1
                if print_errors:
                    print(f"[FAIL] {fpath}:{ln}  expected {correct}, got {assistant_ans}")

    print(f"[large_mat_adv] {passed}/{total} passed, {failed} failed")

# Testing Correctness of `is_legal`

In [5]:
IS_LEGAL_REGEX = re.compile(
    r"Can you legally play\s+([a-h][1-8][a-h][1-8][qrbn]?|[KQRBN]?[a-h]?[1-8]?x?[a-h][1-8](=[QRBN])?\+?#?)\?",
    re.I
)

def _uci_or_san_to_move(board: chess.Board, token: str) -> Optional[chess.Move]:
    """
    Try interpreting *token* as UCI first, else as SAN for the current board.
    Returns a `chess.Move` or None if parsing fails.
    """
    try:
        return chess.Move.from_uci(token)
    except ValueError:
        pass
    try:
        return board.parse_san(token)
    except ValueError:
        return None


def test_is_legal(files: List[str], print_errors: bool = False) -> None:
    """
    Validate 'is_legal' samples: for each JSONL line, confirm the assistant’s
    'Yes' / 'No' matches the true legality of the queried move for the side‑to‑move.
    """
    total = passed = failed = 0

    for fpath in files:
        for ln, line in enumerate(Path(fpath).open(encoding="utf-8"), 1):
            obj           = json.loads(line)
            user_text     = obj["chat"][1][1]
            assistant_ans = obj["chat"][2][1].strip()

            m = IS_LEGAL_REGEX.search(user_text)
            if not m:
                print(f"[WARN] No is_legal question {fpath}:{ln}")
                continue

            move_token = m.group(1)
            board      = chess.Board(visual_to_fen(user_text))
            move       = _uci_or_san_to_move(board, move_token)

            if move is None:
                print(f"[WARN] Unparsable move '{move_token}'  {fpath}:{ln}")
                continue

            is_legal = move in board.legal_moves
            correct  = "Yes" if is_legal else "No"

            total += 1
            if assistant_ans == correct:
                passed += 1
            else:
                failed += 1
                if print_errors:
                    print(f"[FAIL] {fpath}:{ln}  expected {correct}, got {assistant_ans}")

    print(f"[is_legal] {passed}/{total} passed, {failed} failed")

# Testing Correctness of `under_attack`

In [6]:
UA_REGEX = re.compile(
    r"Can your (pawn|knight|bishop|rook|queen|king) take their "
    r"(pawn|knight|bishop|rook|queen|king)\?",
    re.I,
)

# piece‑type ↔ letter maps identical to those in the generator
_piece_letter_map = {
    chess.PAWN: "p", chess.KNIGHT: "n", chess.BISHOP: "b",
    chess.ROOK: "r", chess.QUEEN: "q", chess.KING: "k",
}
_letter_to_piece = {v: k for k, v in _piece_letter_map.items()}
_word_to_letter  = {
    "pawn": "p", "knight": "n", "bishop": "b",
    "rook": "r", "queen": "q", "king": "k",
}

_NONKING_TYPES = {chess.PAWN, chess.KNIGHT, chess.BISHOP,
                  chess.ROOK, chess.QUEEN}


def _capture_pairs(board: chess.Board, side: chess.Color, allow_king: bool) -> set:
    """
    Return {(attacker_letter, victim_letter), ...} for *side* capturing legally.
    `allow_king` determines whether captures of the enemy king count.
    """
    b = board if side == board.turn else board.copy(stack=False)
    b.turn = side
    pairs = set()
    for mv in b.legal_moves:
        if b.color_at(mv.to_square) != (not side):
            continue
        if not allow_king and b.piece_type_at(mv.to_square) == chess.KING:
            continue
        atk = _piece_letter_map[b.piece_type_at(mv.from_square)]
        vic = _piece_letter_map[b.piece_type_at(mv.to_square)]
        pairs.add((atk, vic))
    return pairs


def test_under_attack(files: List[str], print_errors: bool = False) -> None:
    """
    Validate 'under_attack' samples produced by _generate_under_attack.
    """
    total = passed = failed = 0

    for fpath in files:
        for ln, line in enumerate(Path(fpath).open(encoding="utf-8"), 1):
            obj           = json.loads(line)
            user_text     = obj["chat"][1][1]
            assistant_ans = obj["chat"][2][1].strip()

            m = UA_REGEX.search(user_text)
            if not m:
                print(f"[WARN] No under_attack question {fpath}:{ln}")
                continue

            atk_letter = _word_to_letter[m.group(1).lower()]
            vic_letter = _word_to_letter[m.group(2).lower()]

            board = chess.Board(visual_to_fen(user_text))
            you, opp = board.turn, not board.turn

            # determine if non‑king victim pieces exist – generator rule
            opp_has_nonking = any(board.pieces(pt, opp) for pt in _NONKING_TYPES)
            allow_king      = not opp_has_nonking

            pair_exists = (atk_letter, vic_letter) in _capture_pairs(
                board, you, allow_king=allow_king
            )

            correct = "Yes" if pair_exists else "No"

            total += 1
            if assistant_ans == correct:
                passed += 1
            else:
                failed += 1
                if print_errors:
                    print(f"[FAIL] {fpath}:{ln}  expected {correct}, got {assistant_ans}")

    print(f"[under_attack] {passed}/{total} passed, {failed} failed")

# Testing `mat_adv_value`

In [7]:
# ────────────────────────────────────────────────────────────────
# 3) “What is the material advantage for <color>?”
#    (Exact centipawn value, white‑positive convention)
# ────────────────────────────────────────────────────────────────
MAT_ADV_VAL_REGEX = re.compile(
    r"What is the material advantage for (white|black)\?", re.I
)

def test_mat_adv_value(files: List[str], print_errors: bool = False) -> None:
    """
    Verifies that the assistant returns the correct centipawn material
    advantage for the queried colour.

    Expected answer: an *integer* string, e.g. "120" or "-330".
        • Positive means an advantage for White.
        • Negative means an advantage for Black.
    """
    total = passed = failed = 0

    for fpath in files:
        for ln, line in enumerate(Path(fpath).open(encoding="utf-8"), 1):
            obj           = json.loads(line)
            user_text     = obj["chat"][1][1]
            assistant_ans = obj["chat"][2][1].strip()

            m = MAT_ADV_VAL_REGEX.search(user_text)
            if not m:
                print(f"[WARN] No mat_adv_value question {fpath}:{ln}")
                continue

            ask_white = m.group(1).lower() == "white"
            board     = chess.Board(visual_to_fen(user_text))
            cp        = material_cp(board)                     # white‑positive
            correct   = cp if ask_white else -cp

            total += 1
            try:
                got = int(assistant_ans)
                if got == correct:
                    passed += 1
                else:
                    failed += 1
                    if print_errors:
                        print(f"[FAIL] {fpath}:{ln}  expected {correct}, got {got}")
            except ValueError:
                failed += 1
                if print_errors:
                    print(f"[FAIL] {fpath}:{ln}  non‑integer answer '{assistant_ans}'")

    print(f"[mat_adv_value] {passed}/{total} passed, {failed} failed")


# Testing `mobility`

In [9]:
# ────────────────────────────────────────────────────────────────
#  mobility‑count test  (piece + square)
# ────────────────────────────────────────────────────────────────
MOB_Q = re.compile(
    r"How many legal moves does your (pawn|knight|bishop|rook|queen|king) at ([a-h][1-8]) have\?",
    re.I
)
WORD2PT = {
    "pawn":   chess.PAWN,
    "knight": chess.KNIGHT,
    "bishop": chess.BISHOP,
    "rook":   chess.ROOK,
    "queen":  chess.QUEEN,
    "king":   chess.KING,
}

def test_mobility(files: List[str], print_errors: bool = False) -> None:
    total = passed = failed = 0

    for fpath in files:
        for ln, line in enumerate(Path(fpath).open(encoding="utf-8"), 1):
            obj           = json.loads(line)
            user_text     = obj["chat"][1][1]
            assistant_ans = obj["chat"][2][1].strip()

            m = MOB_Q.search(user_text)
            if not m:        # not a mobility‑count prompt
                continue

            ptype_word, sq_str = m.groups()
            ptype   = WORD2PT[ptype_word.lower()]
            square  = chess.SQUARE_NAMES.index(sq_str.lower())

            board   = chess.Board(visual_to_fen(user_text))
            you     = board.turn

            # verify the square really holds that piece for the side to move
            if board.piece_type_at(square) != ptype or board.color_at(square) != you:
                expected = 0
            else:
                expected = sum(1 for mv in board.legal_moves if mv.from_square == square)

            total += 1
            try:
                got = int(assistant_ans)
                if got == expected:
                    passed += 1
                else:
                    failed += 1
                    if print_errors:
                        print(f"[FAIL] {fpath}:{ln}  expected {expected}, got {got}")
            except ValueError:
                failed += 1
                if print_errors:
                    print(f"[FAIL] {fpath}:{ln}  non‑integer answer '{assistant_ans}'")

    print(f"[mobility] {passed}/{total} passed, {failed} failed")


# Testing `cloze_capture`

In [10]:
CLOZE_REGEX = re.compile(
    r"My piece on __ could take the opponent's (\w+) on ([a-h][1-8])\.", re.I
)

_PNAME_TO_PTYPE = {
    "pawn":   chess.PAWN,
    "knight": chess.KNIGHT,
    "bishop": chess.BISHOP,
    "rook":   chess.ROOK,
    "queen":  chess.QUEEN,
    "king":   chess.KING,
}

def test_cloze_capture(files: List[str], print_errors: bool = False) -> None:
    total = passed = failed = 0

    for fpath in files:
        for ln, line in enumerate(Path(fpath).open(encoding="utf-8"), 1):
            obj           = json.loads(line)
            user_text     = obj["chat"][1][1]
            assistant_ans = obj["chat"][2][1].strip().lower()

            m = CLOZE_REGEX.search(user_text)
            if not m:
                continue  # not a cloze-capture prompt

            victim_name, target_sq_str = m.groups()
            target_sq  = chess.parse_square(target_sq_str)
            victim_pt  = _PNAME_TO_PTYPE[victim_name.lower()]

            board = chess.Board(visual_to_fen(user_text))
            if board.piece_type_at(target_sq) != victim_pt:
                if print_errors:
                    print(f"[WARN] Victim mismatch {fpath}:{ln}")
                continue

            # all legal captures by side-to-move that land on target
            caps = [mv for mv in board.legal_moves
                    if mv.to_square == target_sq and board.is_capture(mv)]

            if len(caps) != 1:
                if print_errors:
                    print(f"[WARN] Non-unique capture {fpath}:{ln}")
                continue

            expected = chess.square_name(caps[0].from_square)
            total += 1
            if assistant_ans == expected:
                passed += 1
            else:
                failed += 1
                if print_errors:
                    print(f"[FAIL] {fpath}:{ln}  expected {expected}, got {assistant_ans}")

    print(f"[cloze_capture] {passed}/{total} passed, {failed} failed")

# Testing Files

In [11]:
PARENT_DIR = "latents_train"
TESTS = {
    'is_check': test_is_check,
    'is_legal': test_is_legal,
    'mat_bal': test_mat_bal,
    'large_mat_adv': test_large_mat_adv,
    'under_attack': test_under_attack,
    'mat_adv_value': test_mat_adv_value,
    'win_prob': test_win_prob,
    "mobility": test_mobility,
    "cloze_capture": test_cloze_capture,
}

# Automatically test all our files in the parent dir
for file_path in Path(PARENT_DIR).glob("*.jsonl"):
    for test_name in TESTS:
        if test_name in file_path.name:
            print(f"{file_path.name}:")
            TESTS[test_name]([file_path])
            print(f"\n")
            break
    else:
        print(f"[WARN] No test defined for {file_path.name}.")

latentsft_trainXL_under_attack_50000.jsonl:
[under_attack] 50000/50000 passed, 0 failed


latentsft_trainXL_win_prob_50000.jsonl:
[win_prob] 50000/50000 passed, 0 failed




In [12]:
# Optionally can inspect incorrect answers
# inspect_lines('latent_sft_is_check_10000.jsonl', [12, 47])